In [1]:
import pandas as pd

In [6]:


df_sm_youtube = pd.read_csv("../data/raw/ksm_youtube_comment.csv")
# display(df_sm_youtube.head())
# display(df_sm_youtube.info())
# display(df_sm_youtube.describe())

df_ds_youtube = pd.read_csv("../data/raw/kds_youtube_comments.csv")
# display(df_ds_youtube.head())
# display(df_ds_youtube.info())
# display(df_ds_youtube.describe())

df_sm_youtube_dropped = df_sm_youtube.drop(columns=['Unnamed: 0'])
df_sm_youtube_dropped = df_sm_youtube_dropped.dropna(subset=['text'])
# display(df_sm_youtube_dropped.info())
# print()

df_ds_youtube_dropped = df_ds_youtube.dropna(subset=['parent_id', 'text'])
df_ds_youtube_dropped = df_ds_youtube_dropped.drop(columns=['comment_id', 'parent_id', 'author', 'is_reply'])
df_ds_youtube_dropped = df_ds_youtube_dropped.rename(columns={'video_id': 'videoId',
                                                              'published_at':'published',
                                                              'like_count': 'likeCount'})
# display(df_ds_youtube_dropped.info())

youtube_merged = pd.concat([df_sm_youtube_dropped, df_ds_youtube_dropped])
# display(youtube_merged.info())

sorted_date = youtube_merged.sort_values('published', ascending=False)
# print(sorted_date.info())
sorted_date['published'] = pd.to_datetime(sorted_date['published'])
sorted_date = sorted_date[
    (sorted_date['published'] >= '2026-06-18') &
    (sorted_date['published'] <= '2026-08-14')
]
# display(sorted_date['published'].min())
# display(sorted_date['published'].max())
print(sorted_date.shape)

save_path = "../data/processed/youtube_merged_sample.csv"
sorted_date.to_csv(save_path, encoding='utf-8')

youtube_merged_sample = pd.read_csv(save_path)
df_youtube_merged_sample = youtube_merged_sample.drop(columns=["Unnamed: 0"])
# print(df_youtube_merged_sample.shape)
# display(youtube_merged_sample)
# display(df_youtube_merged_sample)
# display(df_youtube_merged_sample['published'])
df_youtube_merged_sample = df_youtube_merged_sample.dropna(subset=["text"]).copy()
df_youtube_merged_sample["text"] = df_youtube_merged_sample["text"].astype(str)
df_youtube_merged_sample["text_raw"] = df_youtube_merged_sample["text"]

(2702, 4)


In [7]:
import re

# 텍스트 특수문자 정리
print("텍스트 전처리 정리 전")
for i in range(10):
    print(f"[{i}] {df_youtube_merged_sample.iloc[i]['text']}")
print()


def clean_text(text):
    # 링크 제거
    text = re.sub(r"<[^>]+>", " ", text)

    # 유튜브 사용자 이름 제거
    text = re.sub(r"@\S+", " ", text)

    # 텍스트 특수문자 제거
    text = re.sub(r"[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9?\s]", " ", text)

    # 텍스트 반복 문자 제거
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)

    # 공백 정규화
    text = re.sub(r"\s+", " ", text).strip()
    return text


df_youtube_merged_sample["text_clean"] = df_youtube_merged_sample["text_raw"].apply(
    clean_text
)

print("텍스트 특수문자 정리, 반복 문자 정리, 공백 정규화")
for i in range(10):
    print(f"[{i}] {df_youtube_merged_sample.iloc[i]['text_clean']}")

텍스트 전처리 정리 전
[0] 우와 벌써 260 찍으셨군여, 고생 많으셨어여!👍
[1] ​@아무거나-g6k네! 감사합니다 !!!!!!❤
[2] 무조건 방무 바꾸는거는 아니에여 환산에서 스텟효율보고 바꾸는거에여🎉
[3] 방무가 잡옵되는 순간이 에테르넬 풀셋하고 칠흑 풀셋에 유챔까지 전부 하는 고인물 유저기준이에영 챌섭은 버프로 방무가 빠방하니까 챌섭에서는 방무 1줄 추가말고 본섭가서 환산주스텟에서 방무 30퍼랑 마력 9퍼 효율 비교해보고 하시면 되세영 환산보고 바꾸는게 최선이에요
[4] 알겠습니당, 조만간 관련 내용으로 정리해서 가져올게요!
[5] 옵션작은 장신구만 천천히 환불 있을때 하시면 되세영
[6] 보스가 안 깨지시거나 스펙업 수단이 없을 때만 해주시면 될 거예요!
[7] @DDUDUDDUDU 인생 낭비가 아닌 취미가 있나? 뭐든 돈과 시간을 쓸텐데 대체 어떤 취미가 인생 낭비가 아님??
[8] @DDUDUDDUDU사람마다 생각이 다른거죠..😅
[9] 메이플 꾸준히 하기에는 인생낭비임 그냥 방학때 한번씩 하다 몇년에 한번씩 돌아오기좋음

텍스트 특수문자 정리, 반복 문자 정리, 공백 정규화
[0] 우와 벌써 260 찍으셨군여 고생 많으셨어여
[1] 감사합니다
[2] 무조건 방무 바꾸는거는 아니에여 환산에서 스텟효율보고 바꾸는거에여
[3] 방무가 잡옵되는 순간이 에테르넬 풀셋하고 칠흑 풀셋에 유챔까지 전부 하는 고인물 유저기준이에영 챌섭은 버프로 방무가 빠방하니까 챌섭에서는 방무 1줄 추가말고 본섭가서 환산주스텟에서 방무 30퍼랑 마력 9퍼 효율 비교해보고 하시면 되세영 환산보고 바꾸는게 최선이에요
[4] 알겠습니당 조만간 관련 내용으로 정리해서 가져올게요
[5] 옵션작은 장신구만 천천히 환불 있을때 하시면 되세영
[6] 보스가 안 깨지시거나 스펙업 수단이 없을 때만 해주시면 될 거예요
[7] 인생 낭비가 아닌 취미가 있나? 뭐든 돈과 시간을 쓸텐데 대체 어떤 취미가 인생 낭비가 아님??
[8] 생각이 다른거죠
[9] 메이플 꾸준히 하기에는 인생낭비임 그냥

In [8]:
from collections import Counter
from kiwipiepy import Kiwi

kiwi = Kiwi()

text = df_youtube_merged_sample['text_clean'].iloc[0]


def tokenize_text(text):
    tokens = [token.form for token in kiwi.tokenize(text)
            if token.tag.startswith(('NN', 'VA', 'VV'))]
    return tokens

df_youtube_merged_sample["tokens"] = (
    df_youtube_merged_sample["text_clean"]
    .apply(tokenize_text)
)

df_youtube_merged_sample[
    ["text_raw", "text_clean", "tokens"]
].head(10)

print(df_youtube_merged_sample.shape)

(2702, 7)


In [10]:
df_llm = pd.read_csv("../data/processed/youtube_questions_gemini_prefilter.csv")
display(df_llm.head(5))
print(df_llm.shape)

,videoId,text,likeCount,published,text_raw,text_clean,tokens,is_question,question_category,question_confidence,llm_error,classification_source
0,FMUmSNtWlEU,지금 막 시작했는대 메인 퀘 다스킵하기가 안대던대 왜그런건가요,1,2026-08-12 22:58:31+00:00,지금 막 시작했는대 메인 퀘 다스킵하기가 안대던대 왜그런건가요,지금 막 시작했는대 메인 퀘 다스킵하기가 안대던대 왜그런건가요,"['시작', '메인', '퀘', '다스', '킵', '대', '거']",True,시스템,1.0,NaN,NaN
1,mQo4xmzyIgY,근대 지금 메이플 키워도 좋을까요?,0,2026-08-12 20:50:49+00:00,근대 지금 메이플 키워도 좋을까요?,근대 지금 메이플 키워도 좋을까요?,"['근대', '메이플', '키우', '좋']",True,육성_레벨링,1.0,NaN,NaN
2,mQo4xmzyIgY,저어 .. 응애인데요ㅠㅠ 하이퍼가 조금조금 성장하고보면 계속 스탯이 바뀌는데 그때마...,2,2026-08-11 18:55:20+00:00,저어 .. 응애인데요ㅠㅠ 하이퍼가 조금조금 성장하고보면 계속 스탯이 바뀌는데 그때마...,저어 응애인데요ㅠㅠ 하이퍼가 조금조금 성장하고보면 계속 스탯이 바뀌는데 그때마다 초...,"['젓', '애', '하이퍼', '성장', '스탯', '바뀌', '그때', '초기'...",True,육성_레벨링,1.0,NaN,NaN
3,mQo4xmzyIgY,레테스킬에는 방무가 있던데 그래도 따로 방무가필요할까요?,1,2026-08-11 18:37:26+00:00,레테스킬에는 방무가 있던데 그래도 따로 방무가필요할까요?,레테스킬에는 방무가 있던데 그래도 따로 방무가필요할까요?,"['레테', '스킬', '방무', '있', '방', '무', '필요']",True,스펙업,1.0,NaN,NaN
4,9saj1Kg7S1I,일단 280까지 찍고 영상처럼 옵션작하면 되는 건가요??,1,2026-08-11 16:23:24+00:00,일단 280까지 찍고 영상처럼 옵션작하면 되는 건가요??,일단 280까지 찍고 영상처럼 옵션작하면 되는 건가요??,"['찍', '영상', '옵션', '작', '되', '거']",True,육성_레벨링,1.0,NaN,NaN


(344, 12)


In [12]:
df_llm_cp = df_llm.copy()
df_llm_cp = df_llm_cp.drop(columns=['text_raw', 'text_clean', 'tokens', 'llm_error', 'question_confidence', 'is_question'])
# display(df_llm_cp)

llm_save_path = "../data/processed/youtube_question_llm_test.csv"
df_llm_cp.to_csv(llm_save_path, encoding='utf-8')

,videoId,text,likeCount,published,question_category
0,FMUmSNtWlEU,지금 막 시작했는대 메인 퀘 다스킵하기가 안대던대 왜그런건가요,1,2026-08-12 22:58:31+00:00,시스템
1,mQo4xmzyIgY,근대 지금 메이플 키워도 좋을까요?,0,2026-08-12 20:50:49+00:00,육성_레벨링
2,mQo4xmzyIgY,저어 .. 응애인데요ㅠㅠ 하이퍼가 조금조금 성장하고보면 계속 스탯이 바뀌는데 그때마...,2,2026-08-11 18:55:20+00:00,육성_레벨링
3,mQo4xmzyIgY,레테스킬에는 방무가 있던데 그래도 따로 방무가필요할까요?,1,2026-08-11 18:37:26+00:00,스펙업
4,9saj1Kg7S1I,일단 280까지 찍고 영상처럼 옵션작하면 되는 건가요??,1,2026-08-11 16:23:24+00:00,육성_레벨링
...,...,...,...,...,...
205,rEMZwLbgmcw,잼이님 이미 똥블렘에 공21퍼를띄워놨다면 미트라를 킵하는게 나을까요?,1,2026-07-30 12:13:57+00:00,장비_아이템
206,f3DLA1EU8vw,근데 왜 6단계 풀셋전에 7단계올리먄 안되는거에요?,0,2026-07-30 11:41:15+00:00,장비_아이템
207,f3DLA1EU8vw,혹시 영상 ost 뭔지알수있을까욤?,0,2026-07-30 11:39:59+00:00,기타
208,f3DLA1EU8vw,2-6 이랑 2-7이랑 효율 뭐가 더 좋나요,0,2026-07-30 11:18:19+00:00,스킬_6차


In [17]:
display(df_llm_cp.head(10))

,videoId,text,likeCount,published,question_category
0,FMUmSNtWlEU,지금 막 시작했는대 메인 퀘 다스킵하기가 안대던대 왜그런건가요,1,2026-08-12 22:58:31+00:00,시스템
1,mQo4xmzyIgY,근대 지금 메이플 키워도 좋을까요?,0,2026-08-12 20:50:49+00:00,육성_레벨링
2,mQo4xmzyIgY,저어 .. 응애인데요ㅠㅠ 하이퍼가 조금조금 성장하고보면 계속 스탯이 바뀌는데 그때마...,2,2026-08-11 18:55:20+00:00,육성_레벨링
3,mQo4xmzyIgY,레테스킬에는 방무가 있던데 그래도 따로 방무가필요할까요?,1,2026-08-11 18:37:26+00:00,스펙업
4,9saj1Kg7S1I,일단 280까지 찍고 영상처럼 옵션작하면 되는 건가요??,1,2026-08-11 16:23:24+00:00,육성_레벨링
5,mQo4xmzyIgY,템화6.7이면 전투력이 2억이나 나와요?\r\n\r\n패파라서그런가 템환6.7로 1...,1,2026-08-11 13:57:16+00:00,스펙업
6,mQo4xmzyIgY,내실도 해줘,1,2026-08-11 12:31:00+00:00,육성_레벨링
7,mQo4xmzyIgY,선생님의 영상에 도움을 받아\r\n메린이가 마침내 챌린저 달성했어요\r\n그런데 이...,2,2026-08-11 12:20:51+00:00,육성_레벨링
8,oHLCAyjxywA,맑음님은 몇채널인지 공개하라 공개하라 !!!,2,2026-08-11 11:49:27+00:00,기타
9,oHLCAyjxywA,브금이 너무 큰거같은데 나만 그런가..,0,2026-08-11 11:21:59+00:00,시스템
